# MLflow for Agentic AI

MLflow's classical tracking (params/metrics/artifacts/models) is not the right shape for LLM applications: what you actually want is **tracing** - a record of every prompt, tool call, and sub-agent hop inside a single request, plus tokens/cost/latency. This notebook uses [LiteLLM](https://docs.litellm.ai/) as the LLM client (it gives us a unified API to any provider, plus built-in cost accounting) and MLflow Tracing to observe a small multi-agent system built on top of it.

Sections:
1. Setup & connection to the remote MLflow server
2. Autologging: `mlflow.litellm.autolog()` for free
3. Manual/specific tracing: reproducing it by hand (and why you normally wouldn't)
4. Building a multi-agent system: orchestrator -> researcher -> writer, with tools
5. Prompt Registry: versioning the agents' system prompts
6. Agent Versions: registering the pipeline itself as a versioned model
7. Running the pipeline: one MLflow run per user query
8. Sessions: grouping a multi-turn conversation together
9. Multiple queries, compared (cost, tokens, latency, hops)
10. Datasets: building a reusable evaluation set
11. Judges: custom LLM judges and `mlflow.genai.evaluate`

## 1. Setup

In [ ]:
import os
import subprocess
import getpass
import time

import mlflow
import litellm
from mlflow.entities import SpanType
from dotenv import load_dotenv

load_dotenv()

os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["MLFLOW_USERNAME"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["MLFLOW_PASSWORD"]
os.environ["MLFLOW_WORKSPACE"] = "hammer-detox"


MLFLOW_TRACKING_URI = "https://mlflow.tools.ltplabs.net"
EXPERIMENT_NAME = "mlflow_detox_ai"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

# The inline trace-card widget rendered below each trace-producing cell does its own
# unthrottled fetch against the tracking server, outside of with_retry - on this
# shared, rate-limited server that trips "exceeds the maximum queries per second
# limit" constantly. Disable it; traces stay fully accessible via the run links
# printed below and via mlflow.get_trace/search_traces later in the notebook.
mlflow.tracing.disable_notebook_display()

LITELLM_MODEL = f"openai/{os.environ['LITELLM_MODEL']}"
LITELLM_ENDPOINT = os.environ["LITELLM_ENDPOINT"]
LITELLM_API_KEY = os.environ["LITELLM_API_KEY"]


def get_git_commit_hash() -> str:
    try:
        return (
            subprocess.check_output(
                ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
            )
            .decode()
            .strip()
        )
    except Exception:
        return "no-commit"


AUTHOR = getpass.getuser()
GIT_COMMIT = get_git_commit_hash()


def call_llm(messages, tools=None, tool_choice=None):
    return litellm.completion(
        model=LITELLM_MODEL,
        api_base=LITELLM_ENDPOINT,
        api_key=LITELLM_API_KEY,
        messages=messages,
        tools=tools,
        tool_choice=tool_choice,
    )


print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Workspace    : {os.environ['MLFLOW_WORKSPACE']}")
print(f"Experiment   : {experiment.name} ({experiment.experiment_id})")
print(f"LiteLLM model: {LITELLM_MODEL} @ {LITELLM_ENDPOINT}")


def with_retry(func, *args, max_retries=5, base_delay=2, **kwargs):
    """Retry on the tracking server's transient QPS rate-limit errors with exponential backoff."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as exc:
            if "queries per second" in str(exc).lower() and attempt < max_retries - 1:
                wait = base_delay * (2**attempt)
                print(f"Rate-limited by the tracking server, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


## 2. Autologging

One line - `mlflow.litellm.autolog()` - and every `litellm.completion()` call becomes an MLflow **trace** with an `LLM`-type span. It captures the request/response, the resolved model, token usage, latency, and (a nice LiteLLM bonus) **cost** - all without touching the call site.

In [ ]:
mlflow.litellm.autolog()

with mlflow.start_run(run_name="autolog_single_call"):
    response = call_llm(
        [{"role": "user", "content": "In one sentence, what is MLflow?"}]
    )
    autolog_trace_id = mlflow.get_last_active_trace_id()

with_retry(mlflow.flush_trace_async_logging)
print(response.choices[0].message.content)


In [ ]:
trace = with_retry(mlflow.get_trace, autolog_trace_id)
span = trace.data.spans[0]

print(f"Span name : {span.name} ({span.span_type})")
print(f"Model     : {span.attributes.get('mlflow.llm.model')}")
print(f"Tokens    : {span.attributes.get('mlflow.chat.tokenUsage')}")
print(f"Cost      : {span.attributes.get('mlflow.llm.cost')}")


That's everything autolog knows about: **one LLM call in isolation**. It has no concept of "this call was made by the Researcher agent", no aggregate cost across a multi-step task, and can't record anything that isn't an LLM call (a plain Python tool, a retrieval step, business logic). For that we need manual instrumentation.

## 3. Manual / specific tracing

`mlflow.trace` (decorator) and `mlflow.start_span` (context manager) let you create spans around *any* function, not just LLM calls, and attach whatever inputs/outputs/attributes you want. Here we turn autolog off and instrument a single call fully by hand, to see exactly what autolog was doing for us.

In [ ]:
mlflow.litellm.autolog(disable=True)

with mlflow.start_run(run_name="manual_single_call"):
    with mlflow.start_span(name="manual_llm_call", span_type=SpanType.LLM) as manual_span:
        messages = [{"role": "user", "content": "In one sentence, what is LiteLLM?"}]
        manual_span.set_inputs(messages)

        response = call_llm(messages)

        manual_span.set_outputs(response.choices[0].message.content)
        manual_span.set_attributes(
            {
                "model": response.model,
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens,
                "cost_usd": litellm.completion_cost(response),
            }
        )
    manual_trace_id = mlflow.get_last_active_trace_id()

with_retry(mlflow.flush_trace_async_logging)
print(response.choices[0].message.content)


Same information, but every field had to be set explicitly. The upside: full control over what's recorded (redact sensitive inputs, add business-specific attributes, decide sampling) - the downside is the boilerplate. In practice you get the best of both by **leaving autolog on for the raw LLM call and adding manual spans only for the orchestration layer around it** - that's what the rest of this notebook does.

## 4. A small multi-agent system

- **Tools**: `get_weather`, `calculate`, `search_docs` - plain Python functions exposed to the LLM via function calling.
- **Researcher agent**: given a question, decides whether it needs a tool, calls it, and produces a research note.
- **Writer agent**: takes the research note and drafts a final, user-facing answer.
- **Orchestrator**: spawns the Researcher, then the Writer, and returns the final answer.

Every one of these is a span: `AGENT` spans for the three agents, `TOOL` spans for tool executions, and (via autolog) `LLM` spans for every underlying model call - all nested under one trace per user query.

In [ ]:
import json


def get_weather(city: str) -> str:
    fake_weather_db = {
        "lisbon": "22C and sunny",
        "porto": "19C and cloudy",
        "berlin": "15C and rainy",
    }
    return fake_weather_db.get(city.lower(), f"No weather data available for {city}")


def calculate(expression: str) -> str:
    allowed_chars = set("0123456789+-*/(). ")
    if not set(expression) <= allowed_chars:
        return "Error: expression contains disallowed characters"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Error: {exc}"


def search_docs(query: str) -> str:
    knowledge_base = {
        "mlflow": "MLflow is an open-source platform for the ML lifecycle: tracking, models, registry, and GenAI tracing.",
        "optuna": "Optuna is a hyperparameter optimization framework using define-by-run search spaces and samplers like TPE.",
        "litellm": "LiteLLM provides a unified OpenAI-compatible interface to 100+ LLM providers, with built-in cost tracking.",
    }
    for key, value in knowledge_base.items():
        if key in query.lower():
            return value
    return "No relevant documents found."


TOOL_IMPLEMENTATIONS = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_docs": search_docs,
}

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a basic arithmetic expression",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search internal documentation for a topic",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    },
]


def execute_tool(tool_name: str, tool_args: dict) -> str:
    with mlflow.start_span(name=tool_name, span_type=SpanType.TOOL) as span:
        span.set_inputs(tool_args)
        result = TOOL_IMPLEMENTATIONS[tool_name](**tool_args)
        span.set_outputs(result)
    return result


## 5. Prompt Registry

The researcher and writer system prompts used above are hardcoded strings inside the agent functions - fine for a demo, risky in production, since every wording tweak is an untracked code change bundled in with logic changes. MLflow's **Prompt Registry** treats prompts as first-class, versioned artifacts, the same way the Model Registry versions a model: each call to `mlflow.genai.register_prompt()` for a given name creates a new immutable *version*, with a commit message and tags, and agents load a prompt by name (latest, a pinned version, or an alias like `@production`) with `mlflow.genai.load_prompt()` instead of embedding literal strings.

In [ ]:
RESEARCHER_PROMPT_NAME = "detox_researcher_system_prompt"
WRITER_PROMPT_NAME = "detox_writer_system_prompt"

researcher_prompt = mlflow.genai.register_prompt(
    name=RESEARCHER_PROMPT_NAME,
    template=(
        "You are a research assistant. Use the available tools to gather facts "
        "needed to answer the user's question, then summarize what you found in "
        "2-3 sentences. Do not answer directly without checking whether a tool is "
        "relevant."
    ),
    commit_message="Initial researcher system prompt",
    tags={"author": AUTHOR},
)

writer_prompt = mlflow.genai.register_prompt(
    name=WRITER_PROMPT_NAME,
    template=(
        "You are a writer. Turn the research notes into a clear, friendly final "
        "answer for the user. Do not mention that you are an AI agent."
    ),
    commit_message="Initial writer system prompt",
    tags={"author": AUTHOR},
)

print(f"{researcher_prompt.name} v{researcher_prompt.version}")
print(f"{writer_prompt.name} v{writer_prompt.version}")


Both agents below now `load_prompt()` their system message from the registry by name (which resolves to the latest version) instead of hardcoding it. `load_prompt(..., link_to_model=True)` - the default - also records which prompt *version* was resolved by the currently active agent version; that's what powers the "Prompts" tab you'll see on an agent version in the MLflow UI, and it's what the next section relies on.

In [ ]:
mlflow.litellm.autolog()  # LLM spans come from autolog again from here on


@mlflow.trace(span_type=SpanType.AGENT, name="researcher_agent")
def researcher_agent(question: str, run_stats: dict) -> str:
    system_prompt = mlflow.genai.load_prompt(RESEARCHER_PROMPT_NAME)
    messages = [
        {"role": "system", "content": system_prompt.format()},
        {"role": "user", "content": question},
    ]

    for _ in range(3):  # cap the tool-calling loop
        response = call_llm(messages, tools=TOOL_SCHEMAS, tool_choice="auto")
        run_stats["num_llm_calls"] += 1
        run_stats["total_cost_usd"] += litellm.completion_cost(response)
        run_stats["total_tokens"] += response.usage.total_tokens

        message = response.choices[0].message
        if not message.tool_calls:
            return message.content

        messages.append(message.model_dump())
        for tool_call in message.tool_calls:
            tool_args = json.loads(tool_call.function.arguments)
            tool_result = execute_tool(tool_call.function.name, tool_args)
            run_stats["num_tool_calls"] += 1
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result,
                }
            )

    return "Research incomplete: too many tool-calling iterations."


@mlflow.trace(span_type=SpanType.AGENT, name="writer_agent")
def writer_agent(question: str, research_notes: str, run_stats: dict) -> str:
    system_prompt = mlflow.genai.load_prompt(WRITER_PROMPT_NAME)
    messages = [
        {"role": "system", "content": system_prompt.format()},
        {
            "role": "user",
            "content": f"Question: {question}\n\nResearch notes: {research_notes}",
        },
    ]
    response = call_llm(messages)
    run_stats["num_llm_calls"] += 1
    run_stats["total_cost_usd"] += litellm.completion_cost(response)
    run_stats["total_tokens"] += response.usage.total_tokens
    return response.choices[0].message.content


@mlflow.trace(span_type=SpanType.CHAIN, name="orchestrator")
def orchestrator(question: str, run_stats: dict, session_id: str = None, user: str = None) -> str:
    mlflow.update_current_trace(
        tags={
            "user_query": question,
            "agents": "researcher_agent,writer_agent",
            "author": AUTHOR,
            "git_commit": GIT_COMMIT,
            "model": LITELLM_MODEL,
        },
        session_id=session_id,
        user=user,
    )
    run_stats["num_agent_hops"] += 1
    research_notes = researcher_agent(question, run_stats)

    run_stats["num_agent_hops"] += 1
    final_answer = writer_agent(question, research_notes, run_stats)

    return final_answer


## 6. Agent Versions

Every run so far has exercised the same code path. In practice you iterate on an agent - swap a prompt, change the tool set, switch models - and need to compare versions against each other, the same way the Model Registry versions a classical ML model. MLflow does this with `mlflow.create_external_model()`, which creates a **`LoggedModel`** ("external" because its artifacts - our Python code - aren't packaged and uploaded, they just live in this repo), plus `mlflow.set_active_model()`, which links every trace produced while it's active to that specific agent version. Registering another `LoggedModel` under the same name is how you create the next version; the MLflow UI groups `LoggedModel`s by name and lets you compare their linked traces, prompts, and metrics side by side.

In [ ]:
AGENT_NAME = "detox_orchestrator_agent"

agent_v1 = mlflow.create_external_model(
    name=AGENT_NAME,
    model_type="agent",
    params={
        "researcher_prompt_version": researcher_prompt.version,
        "writer_prompt_version": writer_prompt.version,
        "model": LITELLM_MODEL,
    },
    tags={"author": AUTHOR, "git_commit": GIT_COMMIT},
)
mlflow.set_active_model(model_id=agent_v1.model_id)

print(f"Active agent version: {agent_v1.name} ({agent_v1.model_id})")
print(f"Params: {agent_v1.params}")


Now suppose we iterate on the writer prompt - cap it to three sentences so answers stay terse. `register_prompt` creates writer prompt **v2**; registering a new `LoggedModel` under the same agent name creates agent **v2**, whose params point at the new prompt version. From here on, every trace produced while `agent_v2` is active will show up under that version in the UI, separate from the traces produced under `agent_v1` above.

In [ ]:
writer_prompt = mlflow.genai.register_prompt(
    name=WRITER_PROMPT_NAME,
    template=(
        "You are a writer. Turn the research notes into a clear, friendly final "
        "answer for the user, in at most 3 sentences. Do not mention that you are "
        "an AI agent."
    ),
    commit_message="Cap the final answer to 3 sentences",
    tags={"author": AUTHOR},
)

agent_v2 = mlflow.create_external_model(
    name=AGENT_NAME,
    model_type="agent",
    params={
        "researcher_prompt_version": researcher_prompt.version,
        "writer_prompt_version": writer_prompt.version,
        "model": LITELLM_MODEL,
    },
    tags={"author": AUTHOR, "git_commit": GIT_COMMIT},
)
mlflow.set_active_model(model_id=agent_v2.model_id)

agent_versions = with_retry(
    mlflow.search_logged_models,
    experiment_ids=[experiment.experiment_id],
    filter_string=f"name = '{AGENT_NAME}'",
    output_format="list",
)
for i, m in enumerate(sorted(agent_versions, key=lambda m: m.creation_timestamp), start=1):
    print(f"v{i}  {m.model_id}  writer_prompt_version={m.params.get('writer_prompt_version')}")

print(f"\nActive agent version is now: {agent_v2.model_id}")


## 7. Running the pipeline: one MLflow run per user query

Each end-user question becomes one MLflow run. The **trace** (created by the `@mlflow.trace`/autolog machinery above) gives us the step-by-step story of that one request, and - since `agent_v2` is the active model - is automatically linked to agent version v2. The **run-level metrics** we log manually below give us the business view MLflow can never infer on its own: total cost, total tokens, how many agent hops and tool calls it took, and wall-clock latency - the numbers you'd actually chart over time or use to compare two prompt versions.

In [ ]:
def run_agentic_task(question: str, run_name: str, session_id: str = None, user: str = None) -> dict:
    run_stats = {
        "num_llm_calls": 0,
        "num_tool_calls": 0,
        "num_agent_hops": 0,
        "total_cost_usd": 0.0,
        "total_tokens": 0,
    }

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags(
            {
                "author": AUTHOR,
                "git_commit": GIT_COMMIT,
                "model": LITELLM_MODEL,
                "pipeline": "orchestrator_researcher_writer",
                "agent_version_id": mlflow.get_active_model_id(),
            }
        )
        mlflow.log_param("user_query", question)

        start = time.perf_counter()
        answer = orchestrator(question, run_stats, session_id=session_id, user=user)
        latency_seconds = time.perf_counter() - start

        trace_id = mlflow.get_last_active_trace_id()

        mlflow.log_metrics(
            {
                "latency_seconds": latency_seconds,
                "total_cost_usd": run_stats["total_cost_usd"],
                "total_tokens": run_stats["total_tokens"],
                "num_llm_calls": run_stats["num_llm_calls"],
                "num_tool_calls": run_stats["num_tool_calls"],
                "num_agent_hops": run_stats["num_agent_hops"],
            }
        )
        mlflow.log_table(
            data={"role": ["user", "assistant"], "content": [question, answer]},
            artifact_file="conversation.json",
        )
        mlflow.set_tag("trace_id", trace_id)

    return {"question": question, "answer": answer, **run_stats, "latency_seconds": latency_seconds}


result = run_agentic_task(
    "What's the weather like in Lisbon, and what is MLflow?", "agentic_task_weather_mlflow"
)
print(result["answer"])


## 8. Sessions

A single trace is one request/response. A chat product is a sequence of them, and you want to look at that sequence as one conversation - to see whether the agent stayed consistent across turns, or to replay what a specific user actually experienced. `mlflow.update_current_trace(session_id=..., user=...)` (already wired into `orchestrator` above) stamps each trace with a `mlflow.trace.session` metadata key; the MLflow UI groups traces sharing a `session_id` into one conversation view, and you can pull them back yourself with `mlflow.search_traces`.

Below, three follow-up questions share one `session_id` to simulate a multi-turn conversation with the same user.

In [ ]:
import uuid

session_id = f"session-{uuid.uuid4().hex[:8]}"
session_user = AUTHOR

conversation_turns = [
    "What is MLflow, in one sentence?",
    "Does it have anything to do with weather data?",
    "Ok, then just tell me the weather in Berlin.",
]

for i, turn in enumerate(conversation_turns):
    run_agentic_task(
        turn,
        f"session_turn_{i}",
        session_id=session_id,
        user=session_user,
    )
    time.sleep(2)  # avoid bursting the shared tracking server's trace-export rate limit

print(f"session_id = {session_id}")


In [ ]:
session_traces = with_retry(
    mlflow.search_traces,
    locations=[experiment.experiment_id],
    filter_string=f"metadata.`mlflow.trace.session` = '{session_id}'",
    order_by=["timestamp_ms ASC"],
)
session_traces[["trace_id", "request_time", "state"]]


## 9. Multiple queries, compared

Run a handful of different questions through the same pipeline, then pull the runs back as a DataFrame - the same `mlflow.search_runs` pattern used for Optuna trials in the classical ML notebook applies here too.

In [ ]:
demo_questions = [
    "What is 15 * 23, and briefly, what is Optuna?",
    "What's the weather in Porto? Also tell me what LiteLLM is.",
    "Tell me about MLflow and calculate 100 / 4.",
]

for i, question in enumerate(demo_questions):
    run_agentic_task(question, f"agentic_task_{i}")
    time.sleep(2)  # avoid bursting the shared tracking server's trace-export rate limit


In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.pipeline = 'orchestrator_researcher_writer'",
    order_by=["start_time DESC"],
)

comparison_cols = [
    "tags.mlflow.runName",
    "params.user_query",
    "metrics.total_cost_usd",
    "metrics.total_tokens",
    "metrics.num_llm_calls",
    "metrics.num_tool_calls",
    "metrics.num_agent_hops",
    "metrics.latency_seconds",
]
runs_df[comparison_cols]


## 10. Datasets

Ad-hoc questions typed into a notebook are fine for exploration, but they aren't reusable: every time you tweak the prompt or swap models you'd retype them, and you'd have no fixed ground truth to compare against. MLflow's **Evaluation Datasets** (`mlflow.genai.datasets`) solve this: a dataset is a named, versioned collection of records - each an `inputs` dict plus optional `expectations` (ground truth) - stored on the tracking server and tied to one or more experiments. Datasets are the input to `mlflow.genai.evaluate()` in the next section, and they're what you'd wire into a CI job to catch regressions before they ship.

In [ ]:
from mlflow.genai.datasets import create_dataset

DATASET_NAME = "detox_agentic_eval_questions"

eval_dataset = create_dataset(
    name=DATASET_NAME,
    experiment_id=experiment.experiment_id,
    tags={"purpose": "regression_testing", "pipeline": "orchestrator_researcher_writer"},
)

eval_dataset = eval_dataset.merge_records(
    [
        {
            "inputs": {"question": "What is MLflow, in one sentence?"},
            "expectations": {
                "expected_facts": ["MLflow is an open-source platform for the ML lifecycle"]
            },
        },
        {
            "inputs": {"question": "What's the weather in Porto?"},
            "expectations": {"expected_facts": ["Porto is 19C and cloudy"]},
        },
        {
            "inputs": {"question": "What is 15 * 23?"},
            "expectations": {"expected_facts": ["The result is 345"]},
        },
    ]
)

print(f"Dataset: {eval_dataset.name} ({eval_dataset.dataset_id})")
eval_dataset.to_df()[["inputs", "expectations"]]


## 11. Judges

A judge is an LLM prompted to *score* another LLM's output - the way you'd automate "does this answer look right?" at a scale no human reviewer could match. `mlflow.genai.make_judge()` builds one from natural-language instructions with template variables (`{{ inputs }}`, `{{ outputs }}`, `{{ expectations }}`, `{{ trace }}`); MLflow also ships built-in judges (`Correctness`, `Safety`, `RelevanceToQuery`, ...) for common criteria. Judges are `Scorer`s, so they plug directly into `mlflow.genai.evaluate()`, which runs a `predict_fn` over every row of a dataset, scores each output, and logs the aggregate results as an MLflow run - the GenAI equivalent of `cross_val_score` for an LLM pipeline.

Since our LLM traffic goes through an internal LiteLLM gateway rather than a provider's public API, the judge needs the same `base_url`/auth wiring as `call_llm` - see the `base_url`/`OPENAI_API_KEY` setup below.

In [ ]:
from mlflow.genai.judges import make_judge
from typing import Literal

# make_judge's OpenAI-compatible path resolves its API key from the OPENAI_API_KEY
# env var (not a constructor argument), treats `base_url` as the full
# chat-completions endpoint rather than just a host, and - unlike litellm.completion,
# which we prefixed with "openai/" in LITELLM_MODEL for its own routing - expects the
# bare model name after "openai:/". All three are quirks of routing a custom gateway
# through it.
os.environ["OPENAI_API_KEY"] = LITELLM_API_KEY
JUDGE_CHAT_URL = LITELLM_ENDPOINT.rstrip("/") + "/chat/completions"
JUDGE_MODEL_NAME = LITELLM_MODEL.split("/", 1)[-1]

conciseness_judge = make_judge(
    name="conciseness",
    instructions=(
        "Evaluate whether the response in {{ outputs }} concisely answers the "
        "question in {{ inputs }}, without unnecessary padding. Respond yes or no."
    ),
    model=f"openai:/{JUDGE_MODEL_NAME}",
    feedback_value_type=Literal["yes", "no"],
    base_url=JUDGE_CHAT_URL,
)

correctness_judge = make_judge(
    name="fact_coverage",
    instructions=(
        "Given the question in {{ inputs }}, the response in {{ outputs }}, and the "
        "expected facts in {{ expectations }}, rate how well the response covers the "
        "expected facts on a 1-5 scale (5 = all facts present and accurate)."
    ),
    model=f"openai:/{JUDGE_MODEL_NAME}",
    feedback_value_type=int,
    base_url=JUDGE_CHAT_URL,
)

test_feedback = conciseness_judge(
    inputs={"question": "What is MLflow?"},
    outputs="MLflow is an open-source platform for the ML lifecycle.",
)
print(test_feedback.value, "-", test_feedback.rationale)


`mlflow.genai.evaluate()` ties it together: it runs `predict_fn` (here, the orchestrator) over every row of `eval_dataset`, captures a trace per row, scores each trace with both judges, and logs the whole thing as one MLflow run with per-row assessments and aggregate metrics.

In [ ]:
def predict_fn(question: str) -> str:
    run_stats = {
        "num_llm_calls": 0,
        "num_tool_calls": 0,
        "num_agent_hops": 0,
        "total_cost_usd": 0.0,
        "total_tokens": 0,
    }
    return orchestrator(question, run_stats)


eval_result = with_retry(
    mlflow.genai.evaluate,
    data=eval_dataset,
    scorers=[conciseness_judge, correctness_judge],
    predict_fn=predict_fn,
)
eval_result.result_df[
    ["request", "response", "conciseness/value", "fact_coverage/value"]
]


## Recap: the MLflow GenAI toolkit, end to end

| Capability | API | What it gives you |
|---|---|---|
| Autologging | `mlflow.litellm.autolog()` | Free `LLM` spans (prompt, response, tokens, cost, latency) for every raw call |
| Manual tracing | `mlflow.trace` / `start_span` | Spans around anything - agents, tools, business logic - nested into one trace per request |
| Prompt Registry | `mlflow.genai.register_prompt` / `load_prompt` | Versioned, reviewable prompts instead of hardcoded strings |
| Agent Versions | `mlflow.create_external_model` / `set_active_model` | A `LoggedModel` per agent iteration, with every trace linked to the version that produced it |
| Runs & metrics | `mlflow.start_run`, `log_metrics` | Business-level aggregates (cost, tokens, hops) charted or compared over time |
| Sessions | `mlflow.update_current_trace(session_id=...)` | Multiple traces grouped into one multi-turn conversation |
| Datasets | `mlflow.genai.create_dataset` | Reusable, versioned question/expectation sets for regression testing |
| Judges | `mlflow.genai.make_judge`, `mlflow.genai.evaluate` | LLM-graded scoring of agent outputs against a dataset, logged as a run |

In practice: **keep autolog on** for the LLM-call layer, **register prompts and agent versions** as you iterate so every trace is attributable to a specific prompt/model combination, **tag sessions** for anything conversational, and **build a dataset + judges** as soon as you have more than a handful of manual test questions - that's the harness you'll actually run in CI.